# Bài 2.3–2.8 · Ranh giới quyết định của các model cổ điển

**Câu hỏi của buổi này:** mỗi model cổ điển chia mặt phẳng ra sao, và từng tham số (C, k, độ sâu, số cây, gamma) đổi hình ranh giới thế nào, kéo theo train và test accuracy ra sao?

Dữ liệu 2D để vẽ được: **hai nửa vầng trăng** (`make_moons`, 600 điểm có nhiễu) và **Wine** (UCI, 178 chai rượu vang của 3 nhà sản xuất, chỉ lấy hai cột `proline` và `color_intensity`). Mọi số đo là accuracy trên phần test 30% chưa dùng để học.

| Mục | Bài | Sẽ thấy |
|---|---|---|
| 1. Dữ liệu | | hai tập điểm, phần train và test |
| 2. Logistic regression | 2.3 | ranh giới thẳng, thêm đặc trưng bậc 3, softmax ba lớp, tham số C |
| 3. kNN | 2.4 | k = 1, 15, 75; quên chuẩn hóa thì ranh giới thành sọc |
| 4. Cây quyết định | 2.5 | độ sâu 1, 5, không giới hạn; luật của cây |
| 5. Random forest | 2.6 | một cây so với 200 cây; accuracy theo số cây, OOB |
| 6. SVM | 2.8 | kernel tuyến tính và RBF, gamma, C, support vector |
| 7. Đặt cạnh nhau | | năm model trên ba tập dữ liệu |
| 8. Tóm tắt | | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Chỉ cần CPU; chạy hết khoảng 1 phút. Bài giảng: [bài 2.3](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.3), [2.4](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.4), [2.5](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.5), [2.6](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.6), [2.8](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.8).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import sklearn
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons, make_circles, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

table(pd.DataFrame([('scikit-learn', sklearn.__version__)], columns=['thành phần', 'phiên bản']), 'Môi trường chạy')
CM = ListedColormap([C['blue'], C['clay'], C['sand']])
CM_LIGHT = ListedColormap(['#c9d8ec', '#f0cfc2', '#ecdcb6'])

def split(X, y): return train_test_split(X, y, test_size=.3, random_state=0, stratify=y)
def acc(m, X, y): return (m.predict(X) == y).mean()

def boundary(ax, m, data, title='', show_test=True, sv=None):
    Xtr, Xte, ytr, yte = data
    X = np.vstack([Xtr, Xte]); pad = (X.max(0) - X.min(0)) * .08
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - pad[0], X[:, 0].max() + pad[0], 300), np.linspace(X[:, 1].min() - pad[1], X[:, 1].max() + pad[1], 300))
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=CM_LIGHT, levels=np.arange(-.5, 3))
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, cmap=CM, vmin=0, vmax=2, s=14, edgecolors='white', linewidths=.4)
    if show_test: ax.scatter(Xte[:, 0], Xte[:, 1], c=yte, cmap=CM, vmin=0, vmax=2, s=22, marker='^', edgecolors='black', linewidths=.4)
    if sv is not None: ax.scatter(sv[:, 0], sv[:, 1], s=60, facecolors='none', edgecolors='black', linewidths=.8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(f'{title}\ntrain {acc(m, Xtr, ytr):.2f} · test {acc(m, Xte, yte):.2f}', fontsize=10)

## 1. Dữ liệu

Chấm tròn là train, tam giác viền đen là test. Wine được chuẩn hóa (trừ trung bình, chia độ lệch chuẩn tính trên train) trừ khi ghi khác: `proline` có giá trị cỡ hàng trăm tới hơn 1 600, còn `color_intensity` chỉ từ 1 tới 13.

In [ ]:
Xm, ym = make_moons(600, noise=.2, random_state=0)
moons = split(Xm, ym)
wine = load_wine(as_frame=True)
Xw_raw = wine.data[['proline', 'color_intensity']].values; yw = wine.target.values
wine_raw = split(Xw_raw, yw)
sc = StandardScaler().fit(wine_raw[0])
wine_s = (sc.transform(wine_raw[0]), sc.transform(wine_raw[1]), wine_raw[2], wine_raw[3])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (data, name) in zip(axes, [(moons, 'hai nửa vầng trăng, 2 lớp'), (wine_raw, 'Wine: proline và color_intensity, 3 lớp (chưa chuẩn hóa)')]):
    Xtr, Xte, ytr, yte = data
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, cmap=CM, vmin=0, vmax=2, s=16, edgecolors='white', linewidths=.4)
    ax.scatter(Xte[:, 0], Xte[:, 1], c=yte, cmap=CM, vmin=0, vmax=2, s=26, marker='^', edgecolors='black', linewidths=.4)
    ax.set_title(f'{name}\n{len(Xtr)} train · {len(Xte)} test', fontsize=10)
axes[1].set(xlabel='proline', ylabel='color_intensity'); axes[0].set(xlabel='x₁', ylabel='x₂')
plt.tight_layout(); plt.show()
rng_tab = pd.DataFrame({'cột': ['proline', 'color_intensity'], 'nhỏ nhất': Xw_raw.min(0), 'lớn nhất': Xw_raw.max(0), 'độ lệch chuẩn': Xw_raw.std(0)})
table(rng_tab, 'Wine: hai cột chênh nhau hàng trăm lần về thang đo', fmt={'nhỏ nhất': '{:.1f}', 'lớn nhất': '{:.1f}', 'độ lệch chuẩn': '{:.1f}'})
check('mỗi lớp có mặt ở cả train và test', all(set(np.unique(d[2])) == set(np.unique(d[3])) for d in (moons, wine_raw)))

## 2. Logistic regression (bài 2.3)

Logistic regression tính một tổ hợp tuyến tính rồi qua sigmoid, nên ranh giới (nơi xác suất bằng 0.5) là **đường thẳng**. Muốn ranh giới cong, thêm đặc trưng: các tích $x_1^a x_2^b$ tới bậc 3. Với ba lớp, softmax cho mỗi lớp một hàm tuyến tính; ranh giới là nơi hai hàm bằng nhau. `C` là nghịch đảo độ mạnh của regularization: C nhỏ thì phạt trọng số lớn mạnh hơn.

In [ ]:
lr_cfg = [('tuyến tính', make_pipeline(StandardScaler(), LogisticRegression())),
          ('bậc 3, C = 0.01', make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(C=.01, max_iter=5000))),
          ('bậc 3, C = 100', make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(C=100, max_iter=5000)))]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
lr_rows = []
for ax, (n, m) in zip(axes, lr_cfg):
    m.fit(moons[0], moons[2]); boundary(ax, m, moons, f'moons · {n}'); lr_rows.append(('moons', n, acc(m, moons[0], moons[2]), acc(m, moons[1], moons[3])))
soft = LogisticRegression().fit(wine_s[0], wine_s[2])
boundary(axes[3], soft, wine_s, 'Wine · softmax 3 lớp'); lr_rows.append(('Wine', 'softmax 3 lớp', acc(soft, wine_s[0], wine_s[2]), acc(soft, wine_s[1], wine_s[3])))
plt.tight_layout(); plt.show()
lrt = pd.DataFrame(lr_rows, columns=['dữ liệu', 'model', 'train', 'test'])
table(lrt, 'Logistic regression', fmt={'train': '{:.3f}', 'test': '{:.3f}'})
p = soft.predict_proba(wine_s[1])
check('xác suất softmax mỗi dòng cộng lại bằng 1', np.allclose(p.sum(1), 1))
check('softmax có đúng 3 bộ hệ số, mỗi bộ 2 trọng số', soft.coef_.shape == (3, 2))
note(f'Đường thẳng đạt test {lrt.iloc[0, 3]:.2f} trên moons: hai nửa vầng trăng lồng vào nhau nên không đường thẳng nào tách hết. '
     f'Thêm đặc trưng bậc 3 với C = 0.01 (phạt mạnh), trọng số bị ép nhỏ nên ranh giới gần như vẫn thẳng: test {lrt.iloc[1, 3]:.2f}. '
     f'C = 100 để ranh giới uốn theo dữ liệu: train {lrt.iloc[2, 2]:.2f}, test {lrt.iloc[2, 3]:.2f}. '
     + f'Trên Wine, ba hàm tuyến tính của softmax chia mặt phẳng thành ba miền có biên thẳng, test {lrt.iloc[3, 3]:.2f}.')

## 3. kNN (bài 2.4)

kNN không học tham số: một điểm mới lấy nhãn đa số của k điểm train gần nhất. k nhỏ thì ranh giới bám từng điểm (kể cả điểm nhiễu), k lớn thì mượt. Khoảng cách tính trên các cột như chúng đang có, nên **thang đo của cột quyết định ai là "gần"**.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
knn_rows = []
for ax, k in zip(axes[:3], [1, 15, 75]):
    m = KNeighborsClassifier(k).fit(moons[0], moons[2]); boundary(ax, m, moons, f'moons · k = {k}'); knn_rows.append((f'moons, k = {k}', acc(m, moons[0], moons[2]), acc(m, moons[1], moons[3])))
for data, n in [(wine_raw, 'Wine chưa chuẩn hóa, k = 5'), (wine_s, 'Wine đã chuẩn hóa, k = 5')]:
    m = KNeighborsClassifier(5).fit(data[0], data[2]); knn_rows.append((n, acc(m, data[0], data[2]), acc(m, data[1], data[3])))
    if data is wine_raw: boundary(axes[3], m, data, n)
plt.tight_layout(); plt.show()
kt = pd.DataFrame(knn_rows, columns=['model', 'train', 'test'])
table(kt, 'kNN', fmt={'train': '{:.3f}', 'test': '{:.3f}'})
check('k = 1 thuộc lòng train', kt.iloc[0, 1] == 1.0)
note(f'k = 1 đúng {kt.iloc[0, 1]:.0%} train nhưng test {kt.iloc[0, 2]:.2f}: ranh giới vẽ vòng quanh từng điểm nhiễu. k = 15 cho test {kt.iloc[1, 2]:.2f}; k = 75 mượt hơn, test {kt.iloc[2, 2]:.2f}. '
     f'Trên Wine chưa chuẩn hóa, khoảng cách gần như chỉ do <code>proline</code> quyết định nên ranh giới thành những sọc đứng (hình phải): test {kt.iloc[3, 2]:.2f}; chuẩn hóa xong là {kt.iloc[4, 2]:.2f}.')

## 4. Cây quyết định (bài 2.5)

Cây hỏi lần lượt các câu "cột j có nhỏ hơn ngưỡng t không?", nên ranh giới gồm các đoạn **song song với trục**. Độ sâu càng lớn thì càng nhiều ô chữ nhật nhỏ.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
tree_rows = []
for ax, depth in zip(axes[:3], [1, 5, None]):
    m = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(moons[0], moons[2])
    boundary(ax, m, moons, f'moons · độ sâu {depth if depth else "không giới hạn"} ({m.get_n_leaves()} lá)')
    tree_rows.append((f'moons, độ sâu {depth if depth else "không giới hạn"}', m.get_depth(), m.get_n_leaves(), acc(m, moons[0], moons[2]), acc(m, moons[1], moons[3])))
tw = DecisionTreeClassifier(max_depth=2, random_state=0).fit(wine_raw[0], wine_raw[2])
boundary(axes[3], tw, wine_raw, 'Wine · độ sâu 2 (không cần chuẩn hóa)')
tree_rows.append(('Wine, độ sâu 2', tw.get_depth(), tw.get_n_leaves(), acc(tw, wine_raw[0], wine_raw[2]), acc(tw, wine_raw[1], wine_raw[3])))
plt.tight_layout(); plt.show()
tt = pd.DataFrame(tree_rows, columns=['model', 'độ sâu thật', 'số lá', 'train', 'test'])
table(tt, 'Cây quyết định', fmt={'train': '{:.3f}', 'test': '{:.3f}'})
rules = export_text(tw, feature_names=['proline', 'color_intensity'], class_names=['nhà 1', 'nhà 2', 'nhà 3'], decimals=1)
display(HTML(f'<div style="font-weight:600;margin:8px 0 2px">Luật của cây Wine độ sâu 2</div><pre style="font-size:13px;line-height:1.35">{rules}</pre>'))
check('cây không giới hạn độ sâu thuộc lòng train', tt.iloc[2, 3] == 1.0)
note(f'Một câu hỏi (độ sâu 1) chỉ cắt mặt phẳng làm đôi: test {tt.iloc[0, 4]:.2f}. Độ sâu 5: test {tt.iloc[1, 4]:.2f}. Không giới hạn độ sâu, cây mọc {tt.iloc[2, 2]} lá để đúng mọi điểm train, test {tt.iloc[2, 4]:.2f}'
     + (': những ô chữ nhật nhỏ quanh điểm nhiễu không giúp gì ở test. ' if tt.iloc[2, 4] < tt.iloc[1, 4] else '. ')
     + f'Cây chỉ so sánh với ngưỡng nên không cần chuẩn hóa: trên Wine thô, hai câu hỏi đã đúng {tt.iloc[3, 4]:.0%} test, và luật đọc được bằng mắt như ở trên.')

## 5. Random forest (bài 2.6)

Random forest train nhiều cây, mỗi cây trên một mẫu bootstrap và chỉ xét một phần cột ở mỗi lần tách, rồi bỏ phiếu. Mỗi cây vẫn thuộc lòng phần dữ liệu của nó, nhưng chúng thuộc lòng **khác nhau**, nên trung bình lại thì mượt. Điểm out-of-bag (OOB) chấm mỗi điểm train bằng các cây không thấy điểm đó.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={'width_ratios': [1, 1, 1.3]})
one = RandomForestClassifier(1, random_state=0).fit(moons[0], moons[2])
rf = RandomForestClassifier(200, oob_score=True, random_state=0).fit(moons[0], moons[2])
boundary(axes[0], one, moons, 'moons · 1 cây bootstrap'); boundary(axes[1], rf, moons, 'moons · 200 cây')
ns = [1, 2, 5, 10, 20, 50, 100, 200, 500]
curve = []
for n in ns:
    f = RandomForestClassifier(n, oob_score=n >= 20, random_state=0).fit(moons[0], moons[2])     # OOB cần đủ cây mới ổn định
    curve.append((n, acc(f, moons[1], moons[3]), f.oob_score_ if n >= 20 else np.nan))
cv_ = pd.DataFrame(curve, columns=['số cây', 'test', 'OOB'])
axes[2].plot(cv_['số cây'], cv_['test'], 'o-', color=C['clay'], lw=2, label='test accuracy')
axes[2].plot(cv_['số cây'], cv_['OOB'], 's--', color=C['blue'], lw=2, label='OOB accuracy (từ train)')
axes[2].set_xscale('log'); axes[2].set(xlabel='số cây', ylabel='accuracy', title='Thêm cây: tăng nhanh rồi đứng yên'); axes[2].legend(frameon=False)
plt.tight_layout(); plt.show()
check('rừng 200 cây tốt hơn một cây trên test', acc(rf, moons[1], moons[3]) > acc(one, moons[1], moons[3]))
note(f'Một cây trên mẫu bootstrap: test {acc(one, moons[1], moons[3]):.2f}, ranh giới lởm chởm. 200 cây: test {acc(rf, moons[1], moons[3]):.2f}. '
     f'OOB của 200 cây là {rf.oob_score_:.2f}, tính từ chính train mà không cần tách thêm dữ liệu'
     + (', gần với test. ' if abs(rf.oob_score_ - acc(rf, moons[1], moons[3])) < .02 else f', lệch test {rf.oob_score_ - acc(rf, moons[1], moons[3]):+.2f}: với {len(moons[1])} điểm test, mỗi điểm đã là {1 / len(moons[1]):.1%}. ')
     + f'Sau khoảng {int(cv_.loc[cv_["test"].idxmax(), "số cây"])} cây, thêm cây gần như không đổi điểm, chỉ tốn thời gian.')

## 6. SVM (bài 2.8)

SVM tìm ranh giới có **lề rộng nhất**; chỉ các điểm nằm trên hoặc trong lề (support vector, khoanh tròn) quyết định ranh giới. Kernel RBF đo độ giống $e^{-\gamma\|x - x'\|^2}$: gamma lớn thì mỗi điểm chỉ ảnh hưởng một vùng nhỏ, ranh giới vòng quanh từng cụm. C lớn thì phạt nặng điểm nằm sai phía lề.

In [ ]:
svm_cfg = [('tuyến tính, C = 1', SVC(kernel='linear', C=1)), ('RBF, gamma = 0.1', SVC(gamma=.1, C=1)), ('RBF, gamma = 1', SVC(gamma=1, C=1)),
           ('RBF, gamma = 300', SVC(gamma=300, C=1)), ('RBF, gamma = 1, C = 0.05', SVC(gamma=1, C=.05)), ('RBF, gamma = 1, C = 1000', SVC(gamma=1, C=1000))]
fig, axes = plt.subplots(2, 3, figsize=(16, 9.2))
svm_rows = []
for ax, (n, m) in zip(axes.flat, svm_cfg):
    m.fit(moons[0], moons[2]); boundary(ax, m, moons, f'{n} · {len(m.support_)} support vector', show_test=False, sv=m.support_vectors_)
    svm_rows.append((n, len(m.support_), acc(m, moons[0], moons[2]), acc(m, moons[1], moons[3])))
plt.tight_layout(); plt.show()
st_ = pd.DataFrame(svm_rows, columns=['model', 'support vector', 'train', 'test'])
table(st_, f'SVM trên moons ({len(moons[0])} điểm train)', fmt={'train': '{:.3f}', 'test': '{:.3f}'})
check('gamma lớn bám train chặt hơn gamma nhỏ', st_.iloc[3, 2] > st_.iloc[1, 2])
best_svm = st_.loc[st_['test'].idxmax()]
note(f'Kernel tuyến tính cho đường thẳng như logistic (test {st_.iloc[0, 3]:.2f}). RBF gamma 0.1 gần như thẳng ({st_.iloc[1, 3]:.2f}); gamma 300 khoanh từng cụm nhỏ: train {st_.iloc[3, 2]:.2f} mà test {st_.iloc[3, 3]:.2f}. '
     f'C nhỏ cho lề rộng, nhiều điểm thành support vector ({st_.iloc[4, 1]}); C lớn cho lề hẹp, ít support vector hơn ({st_.iloc[5, 1]}). Tốt nhất trên test: {best_svm["model"]} ({best_svm["test"]:.2f}).')

## 7. Đặt cạnh nhau

Năm model với tham số vừa phải, trên ba tập: moons, hai vòng tròn lồng nhau (`make_circles`), và Wine đã chuẩn hóa.

In [ ]:
Xc, yc = make_circles(400, noise=.15, factor=.5, random_state=0)
sets = {'moons': moons, 'vòng tròn': split(Xc, yc), 'Wine': wine_s}
zoo = {'logistic': lambda: LogisticRegression(), 'kNN, k = 15': lambda: KNeighborsClassifier(15), 'cây, độ sâu 4': lambda: DecisionTreeClassifier(max_depth=4, random_state=0),
       'random forest': lambda: RandomForestClassifier(200, random_state=0), 'SVM RBF': lambda: SVC(gamma=1)}
fig, axes = plt.subplots(3, 5, figsize=(20, 11.5))
res = {}
for r, (sn, data) in enumerate(sets.items()):
    for c, (mn, mk) in enumerate(zoo.items()):
        m = mk().fit(data[0], data[2]); boundary(axes[r, c], m, data, f'{sn} · {mn}'); res[(sn, mn)] = acc(m, data[1], data[3])
plt.tight_layout(); plt.show()
cmp_ = pd.Series(res).unstack()[list(zoo)].loc[list(sets)]
table(cmp_.reset_index(names='dữ liệu'), 'Test accuracy', fmt={c: '{:.2f}' for c in zoo})
winners = {s: cmp_.loc[s].idxmax() for s in sets}
check('logistic thua trên vòng tròn (không có đường thẳng nào tách được)', cmp_.loc['vòng tròn', 'logistic'] < .7)
note(f'Không model nào thắng mọi nơi: tốt nhất trên moons là {winners["moons"]}, trên vòng tròn là {winners["vòng tròn"]}, trên Wine là {winners["Wine"]}. '
     f'Logistic chỉ đạt {cmp_.loc["vòng tròn", "logistic"]:.2f} trên vòng tròn vì không đường thẳng nào tách được hai vòng; thêm đặc trưng x₁² + x₂² là giải quyết xong (thử ở cuối).')

## 8. Tóm tắt

In [ ]:
summary = [('logistic trên moons: tuyến tính / bậc 3 (C = 0.01)', f'{lrt.iloc[0, 3]:.2f} / {lrt.iloc[1, 3]:.2f}'),
           ('kNN: k = 1 train / test', f'{kt.iloc[0, 1]:.2f} / {kt.iloc[0, 2]:.2f}'),
           ('kNN trên Wine: chưa / đã chuẩn hóa', f'{kt.iloc[3, 2]:.2f} / {kt.iloc[4, 2]:.2f}'),
           ('cây không giới hạn: số lá, test', f'{tt.iloc[2, 2]} lá, {tt.iloc[2, 4]:.2f}'),
           ('random forest: 1 cây / 200 cây (test)', f'{acc(one, moons[1], moons[3]):.2f} / {acc(rf, moons[1], moons[3]):.2f}'),
           ('SVM RBF gamma 300: train / test', f'{st_.iloc[3, 2]:.2f} / {st_.iloc[3, 3]:.2f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 7 thêm cột $x_1^2 + x_2^2$ cho tập vòng tròn rồi chạy lại logistic; tăng nhiễu của `make_moons` lên 0.5 và xem k tốt nhất của kNN dịch về đâu; ở mục 6 dùng kernel RBF trên Wine chưa chuẩn hóa.